# Clean Public ERP Datasets

This notebook visualises all five **ERP CORE** components (P3, N170, N400, N2pc, MMN)
plus **NOD-EEG** (visual object recognition) using the
same processing pipeline as `data_vis.ipynb`:

- baseline correction on the prestimulus window `-0.2 s .. 0 s`, then post-stimulus segment only
- sorting by every available sort variable (including `reaction_time_ms` / `rt` where present)
- `zscore_timepoints(...)` per time index across trials
- Gaussian low-pass
- resize to `64 x 64`
- asymmetric zero-anchored colorbar

All datasets are **already cleaned** by their original authors — no additional
interpolation or artifact correction is applied here.

### Available sort variables per component

| Component | Continuous | Categorical |
| --------- | ---------- | ----------- |
| **P3** | `reaction_time_ms` | `condition` (Rare/Frequent), `stimulus_code`, `block_target_code`, `trial_stimulus_code`, `is_target_match` |
| **N170** | `reaction_time_ms`, `stimulus_exemplar_index` | `condition` (Faces/Cars/Scrambled\*), `stimulus_family`, `stimulus_code` |
| **N400** | `reaction_time_ms` | `condition` (Related/Unrelated), `stimulus_code` |
| **N2pc** | `reaction_time_ms` | `condition` (Left/Right Target), `stimulus_code` |
| **MMN** | — | `condition`, `stimulus_code` |
| **NOD-EEG** | `rt` (reaction time, s) | `super_class` (30 categories), `stim_is_animate`, `resp_is_right`, `session`, `run` |

### Not yet integrated

- **ZuCo 2.0** (OSF: `osf.io/2urht`) — ICA-cleaned (MARA) fixation-locked EEG, 18 subjects.
  Raw EEG is at sentence-level; extracting word-level fixation-locked epochs requires
  additional fixation-onset alignment (TODO).

In [2]:
import Pkg

week15_dir = if isfile(joinpath(pwd(), "erp_core.ipynb"))
    pwd()
else
    joinpath(pwd(), "notebooks", "week_15")
end

Pkg.activate(joinpath(week15_dir, "..", "model_test"))

using CairoMakie
using DataFrames

include(joinpath(week15_dir, "try_new_data_helpers.jl"))
using .Week15TryNewData

println("Week15 dir: ", week15_dir)
println("Target image size: ", REAL_TARGET_SIZE)

Week15 dir: /home/benjamin/Dokumente/BA2/notebooks/week_15

  Activating project at `~/Dokumente/BA2/notebooks/model_test`



Target image size: (64, 64)


## Load All Clean Bundles

- **ERP CORE** (5 components): official `*_interp_ar.set/.fdt` files (ICA + interpolation + artifact rejection)
- **NOD-EEG**: `.fif` epoch files from OpenNeuro ds005811 (ICA + RANSAC + Zapline + baseline correction, 62 channels, 250 Hz, 4000 trials/subject)

In [ ]:
all_keys = vcat(ERP_CORE_DATASET_KEYS, NEW_PUBLIC_DATASET_KEYS)

# Only load datasets that are actually present on disk
available_keys = String[]
for key in all_keys
    dir = joinpath(DATASETS_ROOT, key)
    if isfile(joinpath(dir, "epochs.hdf5")) && isfile(joinpath(dir, "events.csv")) && isfile(joinpath(dir, "metadata.json"))
        push!(available_keys, key)
    else
        println("  [skip] $key — not yet downloaded")
    end
end

bundles = [load_clean_dataset_bundle(key) for key in available_keys]

external_dataset_summary_df(bundles)

## Available Sort Columns

All non-constant, non-bookkeeping columns from `events.csv`.
`preview_default = true` marks the columns that are auto-plotted
(excludes `sample_index`, `epoch_index`, `source_file`).

In [4]:
available_sort_columns_df(bundles)

Row,dataset_key,component,sort_col,unique_values,value_type,preview_default
,String,String,String,Int64,String,Bool
1,erp_core_p3_clean,P3,reaction_time_ms,360,Float64,true
2,erp_core_p3_clean,P3,condition,2,InlineStrings.String15,true
3,erp_core_p3_clean,P3,trial_stimulus_code,5,Int64,true
4,erp_core_p3_clean,P3,epoch_index,198,Int64,false
5,erp_core_p3_clean,P3,bin_id,2,Int64,true
6,erp_core_p3_clean,P3,bin_label,2,InlineStrings.String31,true
7,erp_core_p3_clean,P3,stimulus_code,25,Int64,true
8,erp_core_p3_clean,P3,block_target_code,5,Int64,true
9,erp_core_p3_clean,P3,is_target_match,2,Bool,true


## Axis Audit

Checks that `(channel, time, trial)` axes match the event table per subject.

In [5]:
dataset_axis_audit_df(bundles)

Row,dataset_key,component,subject_label,observed_axes,expected_axes,tmin_s,tmax_s,event_rows,trial_axis,status
,String,String,String,String,String,Float32,Float32,Int64,Int64,String
1,erp_core_p3_clean,P3,sub-001,"(35, 256, 195)","(35, 256, 195)",-0.199219,0.796875,195,195,ok
2,erp_core_p3_clean,P3,sub-002,"(35, 256, 198)","(35, 256, 198)",-0.199219,0.796875,198,198,ok
3,erp_core_p3_clean,P3,sub-003,"(35, 256, 183)","(35, 256, 183)",-0.199219,0.796875,183,183,ok
4,erp_core_p3_clean,P3,sub-004,"(35, 256, 177)","(35, 256, 177)",-0.199219,0.796875,177,177,ok
5,erp_core_n170_clean,N170,sub-001,"(35, 256, 240)","(35, 256, 240)",-0.199219,0.796875,240,240,ok
6,erp_core_n170_clean,N170,sub-002,"(35, 256, 314)","(35, 256, 314)",-0.199219,0.796875,314,314,ok
7,erp_core_n170_clean,N170,sub-003,"(35, 256, 301)","(35, 256, 301)",-0.199219,0.796875,301,301,ok
8,erp_core_n170_clean,N170,sub-004,"(35, 256, 292)","(35, 256, 292)",-0.199219,0.796875,292,292,ok
9,erp_core_n400_clean,N400,sub-001,"(35, 256, 110)","(35, 256, 110)",-0.199219,0.796875,110,110,ok


## Fixation Reference

A few reference images from the fixation dataset (128 channels, 512 Hz, 2508 trials)
so the visual structure can be compared directly.

In [ ]:
fixation_cache = load_fixation_reference_cache(per_sort_var = 8)
fig_fix = plot_fixation_reference_grid(fixation_cache)
fig_fix

## ERP Image Previews — All Sort Variables

For each component and each preview-default sort variable,
the notebook plots **8 randomly selected (subject, channel) combinations**
in a 2×4 grid.

This includes `reaction_time_ms` for P3, N170, N400, and N2pc
(MMN is a passive paradigm without behavioural responses).

In [ ]:
for bundle in bundles
    specs = recommended_preview_specs(bundle)
    println("\n" * repeat("=", 60))
    println("  ", bundle.dataset_key, " (", String(bundle.metadata.component), ")")
    println("  subjects: ", join(bundle.subject_labels, ", "))
    println("  channels: ", join(bundle.channel_names[1:min(6, length(bundle.channel_names))], ", "),
            length(bundle.channel_names) > 6 ? "..." : "")
    println("  sort variables to plot: ", join([string(s.sort_col) for s in specs], ", "))
    println(repeat("=", 60))

    for spec in specs
        preview = build_dataset_sort_preview(bundle;
            sort_col = spec.sort_col,
            filters  = spec.filters,
            n_samples = 8,
        )
        fig = plot_dataset_sort_preview(preview; n_cols = 4)
        display(fig)
    end
end